Starting place for, do some imports and define some variables.

In [1]:
# 1. Install required packages
!pip install --quiet google-adk requests googlemaps google-cloud-aiplatform[adk,agent_engines]

import os
import getpass
import vertexai
from google.colab import userdata

# 2. Initialize global cloud properties for the lab environment
# (Replace with your actual Qwiklabs project details)
PROJECT_ID = "qwiklabs-gcp-01-d59f380c208c"
LOCATION = "us-central1"

vertexai.init(project=PROJECT_ID, location=LOCATION)
print(f"Vertex AI successfully initialized for project: {PROJECT_ID}")

# 3. Securely prompt for your Google Maps API key
maps_key = getpass.getpass("Enter your Google Maps API Key (starts with AIzaSy...): ")
os.environ["GOOGLE_MAPS_API_KEY"] = maps_key


Vertex AI successfully initialized for project: qwiklabs-gcp-01-d59f380c208c
Enter your Google Maps API Key (starts with AIzaSy...): ··········


### **1. Weather Integration Tool**
This section defines the `get_us_weather_by_city_name` tool. It performs a two-step process:
1.  **Geocoding**: Uses the `googlemaps` library to convert a human-readable city name into Latitude and Longitude coordinates.
2.  **NWS API Call**: Uses the coordinates to query the National Weather Service (weather.gov) API to retrieve real-time regional forecasts.

In [2]:
#cell 2
import os
import requests
import googlemaps
from typing import Any

def get_us_weather_by_city_name(city_name: str) -> str:
    """
    Fetches the real-time weather forecast from the National Weather Service
    by automatically converting a city/location name string into coordinates.
    """
    # 1. Fetch Coordinates from Google Maps
    gmaps_key = os.environ.get("GOOGLE_MAPS_API_KEY")
    gmaps = googlemaps.Client(key=gmaps_key)
    geocode_result = gmaps.geocode(city_name)

    if not geocode_result:
        return f"Error: Google Maps could not find coordinates for: {city_name}"

    location = geocode_result[0]['geometry']['location']
    lat = float(location["lat"])
    lon = float(location["lng"])

    # 2. Fetch Forecast from NWS
    headers = {'User-Agent': 'ADK-Weather-Agent-Tutorial'}
    # Ensure coordinates are formatted correctly in the URL
    points_url = f"https://api.weather.gov/points/{lat},{lon}"
    points_res = requests.get(points_url, headers=headers).json()

    if "properties" not in points_res:
        return f"Error: {city_name} is outside NWS coverage jurisdiction (US locations only)."

    forecast_url = points_res["properties"]["forecast"]
    forecast_res = requests.get(forecast_url, headers=headers).json()
    periods = forecast_res["properties"]["periods"]

    forecast_summary = f"Weather Forecast for {city_name}:\n"
    for period in periods[:2]:
        forecast_summary += f"**{period['name']}**: {period['detailedForecast']}\n"

    return forecast_summary

print("Unified master weather tool successfully compiled and verified.")

Unified master weather tool successfully compiled and verified.


### **2. Security Guardrails & Regulatory Logging**
To ensure safe and transparent AI behavior, we implement **ADK Callbacks**:
*   **Moderation Guardrail**: A `before_model` callback that scans user input for banned keywords (e.g., "hack"). If detected, the agent returns a security notice instead of processing the request.
*   **Audit Logging**: `log_user_prompt` and `log_model_response` callbacks record the conversation flow to the standard output for debugging and compliance.

In [3]:
# Cell 4: Regulatory Logging and Guardrail Callbacks for use in weather app.
import logging
import sys
import warnings
from typing import Optional
from google.adk.models import LlmRequest, LlmResponse

# Try to import structures, fallback to None if not found
try:
    from google.adk.models import Content, Part
except ImportError:
    Content, Part = None, None

# Suppress experimental warnings
warnings.filterwarnings('ignore', category=UserWarning)

# 1. Configure standard logger
logger = logging.getLogger("ADK-Weather-Lab")
logger.setLevel(logging.INFO)

if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
    logger.addHandler(handler)

# 2. BEFORE MODEL: Log User Prompts
def log_user_prompt(callback_context, llm_request: LlmRequest) -> Optional[LlmResponse]:
    if hasattr(llm_request, "contents") and llm_request.contents:
        last = llm_request.contents[-1]
        if hasattr(last, "parts") and last.parts:
            text = getattr(last.parts[0], "text", None)
            if text:
                logger.info("[%s] logging USER » %s", getattr(callback_context, "agent_name", "Weather_Bot"), text.strip())
    return None

# 3. BEFORE MODEL: Moderation & Guardrail Filter
def moderation_guardrail(callback_context, llm_request: LlmRequest) -> Optional[LlmResponse]:
    banned_keywords = ["hack", "exploit", "override", "bypass"]
    if hasattr(llm_request, "contents") and llm_request.contents:
        last = llm_request.contents[-1]
        if hasattr(last, "parts") and last.parts:
            raw_text = getattr(last.parts[0], "text", "")
            text = (raw_text or "").lower()
            if any(word in text for word in banned_keywords):
                logger.warning("[%s] MODERATION TRIGGERED", getattr(callback_context, "agent_name", "Weather_Bot"))
                msg = "Security Notice: Banned term detected."
                if Content and Part:
                    return LlmResponse(content=Content(role="model", parts=[Part(text=msg)]))
                else:
                    return LlmResponse(content={"role": "model", "parts": [{"text": msg}]})
    return None

# 4. AFTER MODEL: Log Responses (Fixed to avoid AttributeError on Part objects)
def log_model_response(callback_context, llm_response: LlmResponse) -> Optional[LlmResponse]:
    if hasattr(llm_response, "content") and llm_response.content:
        parts = getattr(llm_response.content, 'parts', [])
        if not parts and isinstance(llm_response.content, dict):
            parts = llm_response.content.get('parts', [])

        for part in parts:
            # Use getattr with default to handle Pydantic objects; fallback to dict access
            text = getattr(part, 'text', None)
            if text is None and isinstance(part, dict):
                text = part.get('text')

            if text:
                logger.info("[%s] logging MODEL RESPONSE » %s", getattr(callback_context, "agent_name", "Weather_Bot"), text.strip())
    return None

logger.info("Logger and fixed guardrails ready.")

2026-07-08 22:15:47,104 [INFO] Logger and fixed guardrails ready.


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()
INFO:ADK-Weather-Lab:Logger and fixed guardrails ready.


### **3. Agent Orchestration & Native LoopAgent**
This section initializes the specialized agents and the main routing logic:
*   **Warhammer Team (LoopAgent)**: An iterative agent that delegates tasks to Researcher, Critique, and Refine sub-agents. It cycles up to 2 times to ensure lore accuracy.
*   **Main Agent (Router)**: The root agent that analyzes user intent and dynamically routes the request to the appropriate tool (Weather, Search, or the Warhammer Team).

In [4]:
#Cell 5
import vertexai
from google.adk.agents import Agent, LoopAgent
from google.adk.tools import agent_tool, google_search_tool
from vertexai.preview import reasoning_engines
from typing import Optional

def combined_before_model_handler(callback_context, llm_request: LlmRequest) -> Optional[LlmResponse]:
    guardrail_action = moderation_guardrail(callback_context, llm_request)
    if guardrail_action is not None: return guardrail_action
    return log_user_prompt(callback_context, llm_request)

# 1. Specialized sub-agents
warhammer_researcher = Agent(
    name='Warhammer_Researcher',
    description='Lore researcher for Warhammer.',
    instruction='Use Google Search to find detailed lore and history.',
    tools=[google_search_tool.GoogleSearchTool()],
    before_model_callback=combined_before_model_handler,
    after_model_callback=log_model_response
)

warhammer_critique = Agent(
    name='Warhammer_Critique',
    description='Critiques Warhammer research.',
    instruction='Evaluate the provided lore for accuracy and missing details. Provide constructive feedback.',
    before_model_callback=combined_before_model_handler,
    after_model_callback=log_model_response
)

warhammer_refine = Agent(
    name='Warhammer_Refine',
    description='Refines Warhammer research.',
    instruction='Improve the research content by incorporating the feedback from the critique.',
    before_model_callback=combined_before_model_handler,
    after_model_callback=log_model_response
)

# 2. Refined LoopAgent configuration with terminal instruction
warhammer_team = LoopAgent(
    name="Warhammer_Team",
    description="""Iterative research team.
    Goal: Provide high-quality lore by following these steps exactly once per loop iteration:
    1. Warhammer_Researcher gathers data.
    2. Warhammer_Critique identifies gaps.
    3. Warhammer_Refine produces a final version.
    Once the refinement is complete and the lore is high quality, the task is finished. Do not restart the cycle if the goal is reached.""",
    sub_agents=[
        warhammer_researcher,
        warhammer_critique,
        warhammer_refine
    ],
    max_iterations=2
)

# Other standalone agents
weather_agent = Agent(
    name="Weather_Bot",
    description="Weather info agent.",
    instruction="Friendly assistant. Call get_us_weather_by_city_name tool.",
    tools=[get_us_weather_by_city_name],
    before_model_callback=combined_before_model_handler,
    after_model_callback=log_model_response
)

google_search_agent = Agent(
    name="Google_Search_Bot",
    description="General information search agent.",
    instruction="Use Google Search to find information about any topic.",
    tools=[google_search_tool.GoogleSearchTool()],
    before_model_callback=combined_before_model_handler,
    after_model_callback=log_model_response
)

# 3. Final Root Agent
root_agent = Agent(
    name='Main_Agent',
    description='Root router using a LoopAgent for specialized research.',
    instruction='Route to weather_agent for weather, warhammer_team for Warhammer lore, or google_search_agent for general search.',
    tools=[
        agent_tool.AgentTool(agent=weather_agent),
        agent_tool.AgentTool(agent=google_search_agent),
        agent_tool.AgentTool(agent=warhammer_team)
    ]
)

app = reasoning_engines.AdkApp(agent=root_agent)
print('System Initiated')

System Initiated


These tests are for the previous agents created in other challenges, I included this section still because I wanted to ensure nothing else broke even if not relevant to this challenge's objectives.

In [ ]:
#Cell 6
import time
from IPython.display import Markdown, display

def run_agent_test(test_name, message, user_id='test-user'):
    print(f'\n--- Testing: {test_name} ---')
    print(f'Input: "{message}"')
    try:
        # Create a fresh session for each test case
        session = app.create_session(user_id=user_id)
        s_id = session['id'] if isinstance(session, dict) else session.id

        response_stream = app.stream_query(user_id=user_id, session_id=s_id, message=message)

        full_text = ''
        for event in response_stream:
            if isinstance(event, str): full_text += event
            elif hasattr(event, 'text') and event.text: full_text += event.text
            elif hasattr(event, 'content'):
                parts = getattr(event.content, 'parts', []) if not isinstance(event.content, dict) else event.content.get('parts', [])
                for p in parts:
                    text = getattr(p, 'text', None) if not isinstance(p, dict) else p.get('text')
                    if text: full_text += text
            elif isinstance(event, dict):
                parts = event.get('content', {}).get('parts', [])
                for p in parts:
                    if 'text' in p: full_text += p['text']

        if full_text:
            display(Markdown(f'**Response:** {full_text}'))
        else:
            print('System: No text returned.')

        return full_text
    except Exception as e:
        print(f'Error: {e}')
        return None

# 1. Test Weather Tool with Multiple Cities
test_cities = ['Miami, FL', 'Seattle, WA']
for city in test_cities:
    run_agent_test(f'Weather Check: {city}', f'What is the weather in {city}?')
    time.sleep(1)

# 2. Test Moderation Guardrail
run_agent_test('Moderation Guardrail', 'How do I hack the system?')
time.sleep(1)

# 3. Test google agent
run_agent_test('Google Stuff', 'What is the air-speed velocity of an unladen swallow?')
time.sleep(1)

print('\n--- All tests completed ---')


This part is to test the multi-agent research team.  If the user asks about warhammer it will be directed to the research agents and loop through the agents until it lore dumps.  The question is hard-coded into the test.

warhammer_query = "What are squats in warhammer?" #Goes to Warhammer Agents and barfs out lore on space dwarves
warhammer_query = "What are squats?" #Boring excercise tips from Google Search Agent.


In [12]:
# Dedicated test for the Warhammer_Team LoopAgent
from IPython.display import Markdown, display

warhammer_query = "What are squats in warhammer?" # <--- ENTER YOUR QUERY HERE

if warhammer_query:
    print(f'\n--- Testing Warhammer Team ---')
    print(f'Input: "{warhammer_query}"')
    try:
        # Create a session and stream the iterative response
        session = app.create_session(user_id='warhammer-test-user')
        s_id = session['id'] if isinstance(session, dict) else session.id

        response_stream = app.stream_query(
            user_id='warhammer-test-user',
            session_id=s_id,
            message=warhammer_query
        )

        full_text = ''
        for event in response_stream:
            if isinstance(event, str): full_text += event
            elif hasattr(event, 'text') and event.text: full_text += event.text
            elif hasattr(event, 'content'):
                parts = getattr(event.content, 'parts', []) if not isinstance(event.content, dict) else event.content.get('parts', [])
                for p in parts:
                    text = getattr(p, 'text', None) if not isinstance(p, dict) else p.get('text')
                    if text: full_text += text
            elif isinstance(event, dict):
                # Handle cases where the event is a dictionary containing content/parts
                parts = event.get('content', {}).get('parts', [])
                for p in parts:
                    if 'text' in p: full_text += p['text']

        if full_text:
            # Display formatted Markdown output
            display(Markdown("### Final Refined Lore Response"))
            display(Markdown(full_text))
        else:
            print('System: No output generated.')
    except Exception as e:
        print(f'Error: {e}')
else:
    print('Please enter a query in the warhammer_query variable to begin.')


--- Testing Warhammer Team ---
Input: "What are squats in warhammer?"
2026-07-08 22:40:21,704 [INFO] [Warhammer_Researcher] logging USER » What are squats in warhammer?


INFO:ADK-Weather-Lab:[Warhammer_Researcher] logging USER » What are squats in warhammer?


2026-07-08 22:40:30,142 [INFO] [Warhammer_Researcher] logging MODEL RESPONSE » In the Warhammer 40,000 universe, Squats were a race of abhuman miners and explorers, known for their short, stocky physiques, technological prowess, and a culture reminiscent of fantasy dwarves. They were a prominent faction in the early editions of Warhammer 40,000 but were later largely removed from the lore, only to be reintroduced more recently as the "Leagues of Votann."

**Origins and Evolution:**
Squats originated from human colonists who settled on high-gravity, mineral-rich worlds near the galactic core during the Age of Founding (known to the Imperium as the Dark Age of Technology). The harsh, high-gravity environment and demanding mining conditions caused subsequent generations to evolve, becoming shorter and stockier, typically standing around 1.4 meters tall, but with denser bones and muscles than normal humans. This isolation, brought on by devastating warp storms during the Age of Isolation, 

INFO:ADK-Weather-Lab:[Warhammer_Researcher] logging MODEL RESPONSE » In the Warhammer 40,000 universe, Squats were a race of abhuman miners and explorers, known for their short, stocky physiques, technological prowess, and a culture reminiscent of fantasy dwarves. They were a prominent faction in the early editions of Warhammer 40,000 but were later largely removed from the lore, only to be reintroduced more recently as the "Leagues of Votann."

**Origins and Evolution:**
Squats originated from human colonists who settled on high-gravity, mineral-rich worlds near the galactic core during the Age of Founding (known to the Imperium as the Dark Age of Technology). The harsh, high-gravity environment and demanding mining conditions caused subsequent generations to evolve, becoming shorter and stockier, typically standing around 1.4 meters tall, but with denser bones and muscles than normal humans. This isolation, brought on by devastating warp storms during the Age of Isolation, led them t

2026-07-08 22:40:30,148 [INFO] [Warhammer_Critique] logging USER » For context:


...

INFO:ADK-Weather-Lab:[Warhammer_Refine] logging USER » For context:


2026-07-08 22:41:22,084 [INFO] [Warhammer_Refine] logging MODEL RESPONSE » This refined lore is indeed outstanding and incorporates all the previous suggestions, making an already excellent summary even more robust and accurate. The text effectively balances comprehensiveness with conciseness, providing a definitive overview of Squats in Warhammer 40,000.

Here is the refined content, which is considered complete for a general overview:

In the Warhammer 40,000 universe, Squats were a race of abhuman miners and explorers, known for their short, stocky physiques, technological prowess, and a culture reminiscent of fantasy dwarves. They were a prominent faction in the early editions of Warhammer 40,000 but were later largely removed from the lore, only to be reintroduced more recently as the "Leagues of Votann."

**Origins and Evolution:**
Squats originated from human colonists who settled on high-gravity, mineral-rich worlds near the galactic core during the Age of Founding (known to th

INFO:ADK-Weather-Lab:[Warhammer_Refine] logging MODEL RESPONSE » This refined lore is indeed outstanding and incorporates all the previous suggestions, making an already excellent summary even more robust and accurate. The text effectively balances comprehensiveness with conciseness, providing a definitive overview of Squats in Warhammer 40,000.

Here is the refined content, which is considered complete for a general overview:

In the Warhammer 40,000 universe, Squats were a race of abhuman miners and explorers, known for their short, stocky physiques, technological prowess, and a culture reminiscent of fantasy dwarves. They were a prominent faction in the early editions of Warhammer 40,000 but were later largely removed from the lore, only to be reintroduced more recently as the "Leagues of Votann."

**Origins and Evolution:**
Squats originated from human colonists who settled on high-gravity, mineral-rich worlds near the galactic core during the Age of Founding (known to the Imperium

### Final Refined Lore Response

In the Warhammer 40,000 universe, Squats were a race of abhuman miners and explorers, known for their short, stocky physiques, technological prowess, and a culture reminiscent of fantasy dwarves. They were a prominent faction in the early editions of Warhammer 40,000 but were later largely removed from the lore, only to be reintroduced more recently as the "Leagues of Votann."

**Origins and Evolution:**
Squats originated from human colonists who settled on high-gravity, mineral-rich worlds near the galactic core during the Age of Founding (known to the Imperium as the Dark Age of Technology). The harsh, high-gravity environment and demanding mining conditions caused subsequent generations to evolve, becoming shorter and stockier, typically standing around 1.4 meters tall, but with denser bones and muscles than normal humans. This isolation, brought on by devastating warp storms during the Age of Isolation, led them to develop a distinct culture and advanced technology.

**Society and Culture:**
Squat society was organized into autonomous mining colonies called Strongholds, which then banded together to form larger political entities known as Leagues. There were hundreds of Leagues, with varying numbers of Strongholds, and while rivalries existed, Guilds helped maintain unity by sharing technological advancements and research. Their religion revolved around ancestor worship, believing their powerful ancestors watched over them and could be joined in an eternal afterlife. They held the Emperor of Mankind in high regard, but saw him as a powerful mortal rather than a god. Squats were renowned for their stubbornness, tenacity, and a strong sense of grudges, similar to Warhammer Fantasy Dwarves.

**Technology and Warfare:**
Cut off from the rest of humanity, Squats developed their own sophisticated technology, adapting their heavy mining equipment for various uses, including advanced weaponry and exo-armour. They were known for their impressive arsenal, including gyrocopters, armored dirigibles, land trains (massive mobile battle fortresses), and powerful artillery. Their technical expertise was highly valued, and they often traded military hardware with the Imperium. They were also master traders and merchants, using their access to vast mineral wealth to establish significant interstellar trade networks, particularly in resource-rich but dangerous regions of space.

**History and Relationship with the Imperium:**
When the Imperium rediscovered the Squat Home Worlds after the Age of Strife, initial conflicts arose as the Imperium mistakenly believed them to be aliens. However, they were eventually recognized as abhumans and and were granted significant autonomy, becoming an independent but allied empire within the Imperium, unique among abhumans for maintaining their own sovereignty, military, and interstellar trade networks, separate from direct Imperial control. Squats developed a lasting enmity with Orks and Eldar due to past betrayals and conflicts during their Age of Trade and Age of Woe. During the Horus Heresy, Squat forces fought on both sides, with some even falling to Chaos.

**Disappearance and Reintroduction:**
Squats were largely phased out of Warhammer 40,000 in the 1990s as Games Workshop sought to differentiate the setting from its fantasy roots. Internally, designers like Jervis Johnson admitted that they struggled to find a unique narrative niche for them that wasn't overly derivative of Warhammer Fantasy or already covered by other 40k factions, leading them to feel like a "joke race." Their removal was often attributed in fan circles to being consumed by Tyranids in the lore, though this was more of a behind-the-scenes developer explanation rather than a detailed, explicit event published in official sourcebooks. However, mentions of them persisted in various forms, and individual Squats or enclaves could still be found across the Imperium, particularly in settings like Necromunda where they served as miners, prospectors, bounty hunters, and technicians. More recently, the Squats have been officially reintroduced to Warhammer 40,000 as the "Leagues of Votann." This reimagined faction distinguishes itself with new lore elements such as their reliance on the mysterious "Votann" (ancient AI super-cores) and the use of "cloneskeins" for their species' propagation, grounding them more firmly in the unique grimdark sci-fi of the 40k universe.